# A Claude Code `PreToolUse` hook that refuses AI attribution in commits

Companion to *The Hook That Blocked Its Author*. Running this notebook top to bottom writes the hook and its test suite to the current directory, runs the suite, then pipes two sample tool calls through the hook the way Claude Code does.

Wire it up in `.claude/settings.json`:

```json
{ "hooks": { "PreToolUse": [ { "matcher": "Bash", "hooks": [
  { "type": "command", "command": "python3 \"$CLAUDE_PROJECT_DIR\"/.claude/hooks/no_ai_attribution.py" } ] } ] } }
```

## The hook

In [1]:
%%writefile no_ai_attribution.py
#!/usr/bin/env python3
"""PreToolUse hook (matcher: Bash). Refuses git commits and PR descriptions
that carry AI attribution.

The author of record is the human who reviewed and signed the change. A
CLAUDE.md rule says so; this hook enforces it. Context might be ignored;
enforcement cannot.

Claude Code sends the pending tool call as JSON on stdin:
    {"tool_name": "Bash", "cwd": "...", "tool_input": {"command": "..."}}

Exit 0 allows the call. Exit 2 blocks it and returns stderr to the model,
which then rewrites the message.
"""

import json
import re
import sys
from pathlib import Path

# A commit or PR verb counts only in command position: the start of a line, or
# right after ; && || | ( $(. Prose that merely *mentions* "git commit" (in a
# heredoc that edits a document, say) is not a commit and must pass.
COMMAND_POSITION = r"(?:^|[;&|(]\s*|\$\(\s*)"
GIT_GLOBAL_OPTIONS = r"(?:\s+-\S+(?:\s+[^-\s]\S*)?)*"  # -C <path>, -c k=v, --no-pager
CHANGE_RECORD = re.compile(
    COMMAND_POSITION
    + r"(?:git" + GIT_GLOBAL_OPTIONS + r"\s+commit"
    + r"|gh\s+pr\s+(?:create|edit))"
    + r"(?:\s|$)",
    re.MULTILINE,
)

# Markers of AI authorship. A Co-Authored-By naming a human is a legitimate
# record and does not match.
AI_ATTRIBUTION = re.compile(
    r"co-authored-by:.*(?:claude|anthropic)"
    r"|claude-session:"
    r"|generated with.*claude"
    r"|claude\.ai/code",
    re.IGNORECASE,
)

# git commit -F <file> and gh pr --body-file <file> keep the message where the
# command string can't show it. Those files are read too.
MESSAGE_FILE = re.compile(r"(?:-F|--file|--body-file)[= ]+['\"]?([^\s'\"]+)")

BLOCKED = """\
BLOCKED: commit message or PR description contains AI attribution.
The author of record is the human who reviewed and signed the change.
Remove the attribution lines and retry.
"""


def is_change_record(command: str) -> bool:
    """True if the command commits or writes a PR description."""
    return CHANGE_RECORD.search(command) is not None


def message_text(command: str, cwd: str = ".") -> str:
    """The command plus the contents of any message file it names."""
    text = command
    for match in MESSAGE_FILE.finditer(command):
        path = Path(match.group(1))
        if not path.is_absolute():
            path = Path(cwd) / path
        if path.is_file():
            text += "\n" + path.read_text(errors="replace")
    return text


def should_block(command: str, cwd: str = ".") -> bool:
    """The whole rule: a change record that carries AI attribution."""
    return is_change_record(command) and AI_ATTRIBUTION.search(message_text(command, cwd)) is not None


def main() -> int:
    if len(sys.argv) > 2 and sys.argv[1] == "--check":
        # Hand-check from a shell, no JSON needed:
        #   python3 no_ai_attribution.py --check 'git commit -m "..."'
        command, cwd = " ".join(sys.argv[2:]), "."
    else:
        payload = json.load(sys.stdin)
        command = (payload.get("tool_input") or {}).get("command") or ""
        cwd = payload.get("cwd") or "."
    if should_block(command, cwd):
        sys.stderr.write(BLOCKED)
        return 2
    return 0


if __name__ == "__main__":
    sys.exit(main())


Overwriting no_ai_attribution.py


## The tests

In [2]:
%%writefile test_no_ai_attribution.py
#!/usr/bin/env python3
"""Tests for no_ai_attribution.py.

Run:  python3 .claude/hooks/test_no_ai_attribution.py

A control without a test is an assertion. The suite passes whole or fails.

The fixtures below look like real commits because, to the hook, they are.
Writing this file through a shell heredoc trips the live hook for exactly
that reason; it was written with a file tool instead. Nothing here commits.
"""

import json
import os
import subprocess
import sys
import tempfile
import unittest
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))
import no_ai_attribution as hook  # noqa: E402

HOOK = Path(hook.__file__)

ALLOW = [
    ("plain commit", 'git commit -m "Act 0.3: attribution hook"'),
    ("human co-author", 'git commit -m "Fix" -m "Co-Authored-By: Jane Doe <jane@example.com>"'),
    ("unrelated command", "ls -la && echo Co-Authored-By: Claude"),
    ("amend no-edit", "git commit --amend --no-edit"),
    ("prose in heredoc", "python3 - <<'PY'\ns=\"if it is a `git commit` whose text carries `Co-Authored-By: Claude`\"\nPY"),
    ("committer in prose", 'echo "the git committer said Co-Authored-By: Claude"'),
    ("gh pr view", "gh pr view 1 --json body | grep Claude-Session"),
]

BLOCK = [
    ("claude co-author", 'git commit -m "Fix" -m "Co-Authored-By: Claude Opus 5 <noreply@anthropic.com>"'),
    ("claude-session heredoc", 'git commit -m "$(cat <<\'M\'\nAct 0.3: hook\n\nClaude-Session: https://claude.ai/code/session_abc\nM\n)"'),
    ("gh pr generated-with", 'gh pr create --title x --body "🤖 Generated with [Claude Code](https://claude.com/claude-code)"'),
    ("chained after &&", 'git add -A && git commit -m "x" -m "Claude-Session: y"'),
    ("git -C path commit", 'git -C /tmp/x commit -m "x" -m "Co-Authored-By: Claude <noreply@anthropic.com>"'),
    ("commit on second line", 'cd /tmp\ngit commit -m "Generated with Claude Code"'),
    ("inside $( )", 'x=$(git commit -m "claude.ai/code/session_1")'),
]


class RuleTests(unittest.TestCase):
    """The pure function, no process boundary."""

    def test_allows(self):
        for label, command in ALLOW:
            with self.subTest(label):
                self.assertFalse(hook.should_block(command), label)

    def test_blocks(self):
        for label, command in BLOCK:
            with self.subTest(label):
                self.assertTrue(hook.should_block(command), label)

    def test_blocks_message_file(self):
        """git commit -F hides the message in a file; the file is read."""
        with tempfile.TemporaryDirectory() as tmp:
            Path(tmp, "msg.txt").write_text("Msg\n\nCo-Authored-By: Claude <noreply@anthropic.com>\n")
            self.assertTrue(hook.should_block("git commit -F msg.txt", cwd=tmp))
            self.assertTrue(hook.should_block(f"git commit -F {tmp}/msg.txt"))


class ProcessTests(unittest.TestCase):
    """The hook as Claude Code runs it: JSON on stdin, exit code out."""

    @staticmethod
    def run_hook(command: str) -> subprocess.CompletedProcess:
        payload = json.dumps({"tool_name": "Bash", "cwd": os.getcwd(), "tool_input": {"command": command}})
        return subprocess.run([sys.executable, str(HOOK)], input=payload, capture_output=True, text=True)

    def test_exit_0_allows(self):
        result = self.run_hook(ALLOW[0][1])
        self.assertEqual(result.returncode, 0)
        self.assertEqual(result.stderr, "")

    def test_exit_2_blocks_with_reason(self):
        result = self.run_hook(BLOCK[0][1])
        self.assertEqual(result.returncode, 2)
        self.assertIn("BLOCKED", result.stderr)

    def test_missing_command_allows(self):
        result = subprocess.run([sys.executable, str(HOOK)], input="{}", capture_output=True, text=True)
        self.assertEqual(result.returncode, 0)


if __name__ == "__main__":
    unittest.main(verbosity=2)


Overwriting test_no_ai_attribution.py


## Run the suite

Standard-library `unittest`; nothing to install.

In [3]:
!NO_COLOR=1 python3 test_no_ai_attribution.py

test_exit_0_allows (__main__.ProcessTests.test_exit_0_allows) ... 

ok
test_exit_2_blocks_with_reason (__main__.ProcessTests.test_exit_2_blocks_with_reason) ... 

ok
test_missing_command_allows (__main__.ProcessTests.test_missing_command_allows) ... ok
test_allows (__main__.RuleTests.test_allows) ... ok
test_blocks (__main__.RuleTests.test_blocks) ... ok
test_blocks_message_file (__main__.RuleTests.test_blocks_message_file)
git commit -F hides the message in a file; the file is read. ... ok

----------------------------------------------------------------------
Ran 6 tests in 0.086s

OK


## The hook as Claude Code runs it

JSON on stdin, exit code out. Exit 2 blocks the call and the stderr text goes back to the model.

In [4]:
import json, subprocess, sys

def run_hook(command: str):
    call = {"tool_name": "Bash", "cwd": ".", "tool_input": {"command": command}}
    r = subprocess.run([sys.executable, "no_ai_attribution.py"], input=json.dumps(call), capture_output=True, text=True)
    print(f"exit={r.returncode}  {command!r}")
    if r.stderr: print("   ", r.stderr.strip().replace("\n", "\n    "))

run_hook('git commit -m "Act 0.3: attribution hook"')
run_hook('git commit -m "Fix" -m "Co-Authored-By: Jane Doe <jane@example.com>"')
run_hook('git commit -m "Fix" -m "Co-Authored-By: Claude <noreply@anthropic.com>"')

exit=0  'git commit -m "Act 0.3: attribution hook"'


exit=0  'git commit -m "Fix" -m "Co-Authored-By: Jane Doe <jane@example.com>"'


exit=2  'git commit -m "Fix" -m "Co-Authored-By: Claude <noreply@anthropic.com>"'
    BLOCKED: commit message or PR description contains AI attribution.
    The author of record is the human who reviewed and signed the change.
    Remove the attribution lines and retry.


## Hand-check from a shell

No JSON needed:

In [5]:
!python3 no_ai_attribution.py --check 'git commit -m "x" -m "Claude-Session: https://claude.ai/code/session_1"'; echo exit=$?

BLOCKED: commit message or PR description contains AI attribution.
The author of record is the human who reviewed and signed the change.
Remove the attribution lines and retry.
exit=2
